In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

#fixed seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

**Part A**

In [ ]:
#loading data
file = loadmat("C:\\Users\\HP\\Downloads\\Xtrain.mat")
key = [k for k in file if not k.startswith("_")][0]
data = file[key].flatten().astype(np.float32)

#split 80-20 for training-validation
train_raw = data[:int(0.8 * len(data))]
val_raw   = data[int(0.8 * len(data)):]

#normalization to [0, 1] range for better training stability
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_raw.reshape(-1, 1)).flatten()
val_scaled   = scaler.transform(val_raw.reshape(-1, 1)).flatten()

In [ ]:
#sequence generation for one-step-ahead prediction
WINDOW_SIZE = 20
def make_sequences(series, window):
    #input-output pairs where input is a sequence of 'window' past values and output is the next value
    X = []
    y = []
    for i in range(len(series) - window):
        X.append(series[i : i + window])
        y.append(series[i + window])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

#dataset for training and validation
X_train, y_train = make_sequences(train_scaled, WINDOW_SIZE)
X_val,   y_val   = make_sequences(val_scaled,   WINDOW_SIZE)

#conversion to PyTorch tensors
X_train_t = torch.tensor(X_train).unsqueeze(-1)
X_val_t   = torch.tensor(X_val).unsqueeze(-1)
y_train_t = torch.tensor(y_train).unsqueeze(-1)
y_val_t   = torch.tensor(y_val).unsqueeze(-1)

#to manage minibatch training
#no shuffling since it's time series data
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=False)

In [ ]:
#LSTM-based model
class LSTMPredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=64, num_layers=2,
                            batch_first=True, dropout=0.1)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        #LSTM returns output for all time steps
        #we take the last one for prediction
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMPredictor().to(device)
print(model)

In [ ]:
EPOCHS = 200 #max n epochs
LR = 1e-3 #initial learning rate
PATIENCE = 20 #for early stopping if no improvement in validation loss

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4) #L2 regularization to prevent overfitting
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5) #reduce LR if val loss plateaus for 10 epochs

train_losses = []
val_losses = []
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    #training
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad() #clear previous gradients
        loss = criterion(model(xb), yb) #compute current batch loss
        loss.backward() #to compute gradients
        nn.utils.clip_grad_norm_(model.parameters(), 1.0) #gradient clipping to prevent exploding gradients
        optimizer.step() #update model parameters

    model.eval() #no dropout
    with torch.no_grad():
        train_loss = criterion(model(X_train_t.to(device)), y_train_t.to(device)).item()
        val_loss   = criterion(model(X_val_t.to(device)),   y_val_t.to(device)).item()

    scheduler.step(val_loss)
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if epoch % 10 == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch}/{EPOCHS} | Train MSE: {train_loss:.6f} | Val MSE: {val_loss:.6f} | LR: {lr:.6f}")

    #early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'val_loss': val_loss}, "best_model.pt")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (best val MSE: {best_val_loss:.6f})")
            break

In [ ]:
#load the best model for evaluation
checkpoint = torch.load("best_model.pt")
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Best model loaded (Val MSE: {checkpoint['val_loss']:.6f})")

model.eval()
with torch.no_grad():
    val_pred_scaled = model(X_val_t.to(device)).cpu().numpy()

val_pred_orig = scaler.inverse_transform(val_pred_scaled)
y_val_orig    = scaler.inverse_transform(y_val.reshape(-1, 1))

mae  = np.mean(np.abs(val_pred_orig - y_val_orig)) #mean absolute error
mse  = np.mean((val_pred_orig - y_val_orig) ** 2) #mean squared error
rmse = np.sqrt(mse) #root mean squared error
mape = np.mean(np.abs((val_pred_orig - y_val_orig) / (y_val_orig))) * 100 #mean absolute percentage error
print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.6f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.2f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label="Train MSE")
ax1.plot(val_losses, label="Val MSE")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("MSE Loss")
ax1.legend()

n = min(300, len(y_val_orig))
ax2.plot(y_val_orig[:n], label="Real")
ax2.plot(val_pred_orig[:n], label="Predicted", linestyle="--")
ax2.set_title(f"Prediction - MAE={mae:.4f} | MAPE={mape:.2f}%")
ax2.set_xlabel("Time step")
ax2.set_ylabel("Laser measurement")
ax2.legend()

plt.tight_layout()
plt.show()

**Part B**

In [ ]:
def train_and_evaluate(window_size, epochs=100, patience=15):
    #build sequences with the given window size
    X_tr, y_tr = make_sequences(train_scaled, window_size)
    X_v,  y_v  = make_sequences(val_scaled,   window_size)

    #convert to tensors
    X_tr_t = torch.tensor(X_tr).unsqueeze(-1)
    X_v_t  = torch.tensor(X_v).unsqueeze(-1)
    y_tr_t = torch.tensor(y_tr).unsqueeze(-1)
    y_v_t  = torch.tensor(y_v).unsqueeze(-1)

    #no shuffling since it's time series data
    loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=64, shuffle=False)

    #fresh model, optimizer and scheduler for each window size
    m         = LSTMPredictor().to(device)
    crit      = nn.MSELoss()
    opt       = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-4) #L2 regularization to prevent overfitting
    sched     = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=8, factor=0.5) #reduce LR if val loss plateaus

    best      = float('inf')
    pat_count = 0

    for epoch in range(1, epochs + 1):
        #training
        m.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad() #clear previous gradients
            loss = crit(m(xb), yb) #compute current batch loss
            loss.backward() #to compute gradients
            nn.utils.clip_grad_norm_(m.parameters(), 1.0) #gradient clipping to prevent exploding gradients
            opt.step() #update model parameters

        #validation
        m.eval()
        with torch.no_grad():
            val_loss = crit(m(X_v_t.to(device)), y_v_t.to(device)).item()
        sched.step(val_loss)

        #early stopping
        if val_loss < best:
            best      = val_loss
            pat_count = 0
        else:
            pat_count += 1
            if pat_count >= patience:
                break

    return best

In [ ]:
#candidate window sizes to evaluate
WINDOW_SIZES = [5, 10, 15, 20, 30, 40, 50, 75, 100]

results = {}
for ws in WINDOW_SIZES:
    print(f"Testing window_size = {ws:3d} ...", end=" ", flush=True)
    val_mse = train_and_evaluate(ws)
    results[ws] = val_mse
    print(f"Val MSE = {val_mse:.6f}")

#find best window size
best_ws  = min(results, key=results.get)
best_mse = results[best_ws]
print(f"\nBest window size: {best_ws}  (Val MSE = {best_mse:.6f})")

#plot validation MSE vs window size
windows = list(results.keys())
mses    = list(results.values())

plt.figure(figsize=(9, 5))
plt.plot(windows, mses, marker='o', linewidth=2, markersize=7)
plt.axvline(best_ws, color='red', linestyle='--', label=f"Best = {best_ws}")
plt.scatter([best_ws], [best_mse], color='red', zorder=5, s=100)
plt.title("Window Size Tuning — Validation MSE")
plt.xlabel("Window size (past time steps)")
plt.ylabel("Validation MSE")
plt.xticks(windows)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

**Part C**

In [ ]:
#retrain the best model using the best window size found in part b
#we use the training data scaled with the same scaler
X_train_best, y_train_best = make_sequences(train_scaled, best_ws)

X_train_best_t = torch.tensor(X_train_best).unsqueeze(-1)
y_train_best_t = torch.tensor(y_train_best).unsqueeze(-1)

best_loader = DataLoader(TensorDataset(X_train_best_t, y_train_best_t), batch_size=64, shuffle=False)

#fresh model with the best window size
best_model     = LSTMPredictor().to(device)
best_criterion = nn.MSELoss()
best_optimizer = torch.optim.Adam(best_model.parameters(), lr=1e-3, weight_decay=1e-4) #L2 regularization to prevent overfitting
best_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(best_optimizer, patience=10, factor=0.5) #reduce LR if val loss plateaus

best_val_loss_final = float('inf')
patience_counter_final = 0

for epoch in range(1, EPOCHS + 1):
    #training
    best_model.train()
    for xb, yb in best_loader:
        xb, yb = xb.to(device), yb.to(device)
        best_optimizer.zero_grad() #clear previous gradients
        loss = best_criterion(best_model(xb), yb) #compute current batch loss
        loss.backward() #to compute gradients
        nn.utils.clip_grad_norm_(best_model.parameters(), 1.0) #gradient clipping to prevent exploding gradients
        best_optimizer.step() #update model parameters

    best_model.eval() #no dropout
    with torch.no_grad():
        train_loss_final = best_criterion(best_model(X_train_best_t.to(device)), y_train_best_t.to(device)).item()

    best_scheduler.step(train_loss_final)

    if epoch % 10 == 0:
        lr = best_optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch}/{EPOCHS} | Train MSE: {train_loss_final:.6f} | LR: {lr:.6f}")

    #early stopping on training loss since we use all data for training here
    if train_loss_final < best_val_loss_final:
        best_val_loss_final = train_loss_final
        torch.save(best_model.state_dict(), "best_model_final.pt")
        patience_counter_final = 0
    else:
        patience_counter_final += 1
        if patience_counter_final >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

#load best final model
best_model.load_state_dict(torch.load("best_model_final.pt"))
print(f"Final model loaded (best Train MSE: {best_val_loss_final:.6f})")

In [ ]:
#recursive prediction of 200 steps
#seed the window with the last 'best_ws' points of the full scaled data
#this ensures the seed is as close as possible to the start of the test set
N_STEPS = 200
full_scaled = scaler.transform(data.reshape(-1, 1)).flatten()
seed_window = full_scaled[-best_ws:].tolist() #last known values as starting window

recursive_preds_scaled = []
best_model.eval()

for _ in range(N_STEPS):
    #build input tensor from current window
    x_input = torch.tensor(seed_window[-best_ws:], dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(device)
    with torch.no_grad():
        next_val = best_model(x_input).item() #predict next step
    recursive_preds_scaled.append(next_val)
    seed_window.append(next_val) #feed prediction back into the window

#inverse transform to original scale
recursive_preds = scaler.inverse_transform(
    np.array(recursive_preds_scaled, dtype=np.float32).reshape(-1, 1)
).flatten()

#plot recursive predictions
plt.figure(figsize=(12, 5))
#show last 100 training points for context
context = scaler.inverse_transform(train_scaled[-100:].reshape(-1, 1)).flatten()
x_context = np.arange(-100, 0)
x_future  = np.arange(0, N_STEPS)
plt.plot(x_context, context,         label="Training data (last 100 points)", color="steelblue")
plt.plot(x_future,  recursive_preds, label="Recursive prediction (200 steps)", color="orange", linestyle="--")
plt.axvline(0, color="gray", linestyle=":", linewidth=1)
plt.title(f"Recursive 200-Step Prediction (window={best_ws})")
plt.xlabel("Time step")
plt.ylabel("Laser measurement")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

**Part D**

In [ ]:
#load the test data released on May 8
#make sure Xtest.mat is downloaded from Blackboard and placed in the Downloads folder
file_test = loadmat("C:\\Users\\HP\\Downloads\\Xtest.mat")
key_test  = [k for k in file_test if not k.startswith("_")][0]
test_data = file_test[key_test].flatten().astype(np.float32)
print(f"Test data loaded: {len(test_data)} samples")

#the test set has exactly 200 points, matching our recursive predictions
#no need to re-scale test data for comparison since we inverse-transformed predictions
test_orig = test_data[:N_STEPS]

#compute MAE and MSE between recursive predictions and real test values
mae_test = np.mean(np.abs(recursive_preds - test_orig)) #mean absolute error
mse_test = np.mean((recursive_preds - test_orig) ** 2)  #mean squared error
print(f"Test MAE : {mae_test:.4f}")
print(f"Test MSE : {mse_test:.6f}")

#plot predicted vs real test values
plt.figure(figsize=(12, 5))
plt.plot(test_orig,       label="Real test values",  color="steelblue", linewidth=1.5)
plt.plot(recursive_preds, label="Predicted values",  color="orange", linestyle="--", linewidth=1.5)
plt.title(f"Predicted vs Real — Test Set\nMAE={mae_test:.4f}  |  MSE={mse_test:.6f}")
plt.xlabel("Time step")
plt.ylabel("Laser measurement")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()